### CHECK AND MAKE SURE TO LOAD THE DATA AFTER THE LAST LOAD DATE

In [0]:
# Đây chỉ là DataFrame
df = spark.sql("""SELECT max(order_date) AS last_load_date FROM datamodeling.default.source_data""")
print(type(df))

# Spark chỉ chạy SQL khi gặp Action (df.collect(); df.first(); df.show())
# Ta dùng df.first() vì nó chỉ trả về dòng đầu tiên (trước đó ta đã dùng max) -> Tiết kiệm bộ nhớ
last_load_date = df.first()['last_load_date']
print(type(last_load_date))

# DataFrame -> first() -> Row(last_load_date=...) -> ["last_load_date"] -> value

<class 'pyspark.sql.connect.dataframe.DataFrame'>
<class 'datetime.date'>


In [0]:
# Lấy ngày mới nhất trong bảng bronze, đối chiếu với source. Nếu có ngày mới hơn thì lấy thêm
if spark.catalog.tableExists("datamodeling.bronze.bronze_source"):
    last_load_date = spark.sql("""SELECT max(order_date) as last_load_date FROM datamodeling.bronze.bronze_source""").first()['last_load_date']

else:
    last_load_date = '1900-01-01'           # dynamic solution

In [0]:
spark.sql(f"""SELECT * FROM datamodeling.default.source_data            
WHERE order_date > '{last_load_date}'""").createOrReplaceTempView("bronze_table")      # create a view on top of selected data

# Taking the latest inserted data, if exist 

In [0]:
%sql
-- Testing if the view exists
SELECT * FROM bronze_table

order_id,order_date,customer_id,customer_name,customer_email,product_id,product_name,product_category,quantity,unit_price,payment_type,country,last_updated
1004,2024-07-02,4,David Kim,david@hotmail.com,504,Samsung Galaxy S23,Electronics,1,799.99,Apple Pay,USA,2024-07-02
1005,2024-07-02,5,Eve Lee,eve@gmail.com,505,Apple Watch,Electronics,1,399.99,Google Pay,USA,2024-07-02


In [0]:
%sql
-- Create the bronze_table for the next phase
CREATE OR REPLACE TABLE datamodeling.bronze.bronze_table 
AS 
SELECT * FROM bronze_source

num_affected_rows,num_inserted_rows
